### 211. Design Add and Search Words Data Structure

1. addWord(word) 的複雜度
   - 時間複雜度：$O(N)$
     - 只需要遍歷長度為 $N$ 的單字一次。在每一層中，我們在 Hash Map（Python 的 dict）中查找或插入字元的時間是 $O(1)$。因此，整體時間與單字長度 $N$ 成線性關係。
   - 空間複雜度（額外空間）：$O(N)$
     - 最壞情況下，如果新插入的單字與樹中現有的單字沒有任何共同前綴，我們需要為這個單字建立 $N$ 個新的 TrieNode。
2. search(word) 的複雜度
   - 時間複雜度
     - 最好情況（無萬用字元 .）：$O(N)$
       - 如果字串是一般單字，不包含 "."，我們只需順著字典樹的節點一路往下找 $N$ 次，每次雜湊表查詢為 $O(1)$。
     - 最壞情況（全為萬用字元 .）：$O(\min(M \times L, \; \Sigma^N))$
       - 當搜尋字串全是 "." 時（例如 "...."），DFS 必須窮舉所有可能的子節點。
         - 從樹的結構上限來看：它最多只能遍歷整棵樹的所有節點，整棵樹的節點數上限為 $O(M \times L)$。
         - 從遞迴樹的展開上限來看：第一層有 $\Sigma$ 個分支，第二層有 $\Sigma^2$ 個分支，直到第 $N$ 層有 $\Sigma^N$ 個分支。
       - 因此，最壞時間複雜度會被這兩個數值同時制約，寫成 $O(\min(M \times L, \; 26^N))$ 。
   - 空間複雜度（額外空間 / Auxiliary Space）：$O(N)$
     - 搜尋時使用的額外空間主要來自 DFS 的系統遞迴棧（Call Stack）。因為樹的最大搜尋深度受限於單字長度 $N$，所以隨時在記憶體中的遞迴棧深度最多為 $O(N)$。
3. 整體資料結構的空間複雜度 (Space Complexity of the Data Structure)空間複雜度：$O(M \times L \times \Sigma)$
   - 在最壞情況下，所有插入的 $M$ 個單字彼此完全沒有重複的前綴，總共會產生 $M \times L$ 個節點。而每個節點內部的 Hash Map 最多會儲存 $\Sigma = 26$ 個字母的指標。

- $N$：目前操作的單字（word）長度。
- $M$：字典樹中已插入的單字總數。
- $L$：字典樹中最長單字的長度。
- $\Sigma$（Sigma）：字元集大小。因為題目限定為英文小寫字母，所以 $\Sigma = 26$。

In [1]:
# 定義 Trie 的節點結構
class TrieNode:
    def __init__(self):
        self.children = {}  # 使用字典儲存每個字元對應的子節點
        self.end = False    # 標記是否為一個單詞的結尾

# 定義 Trie 主結構
class WordDictionary:

    def __init__(self):
        self.root = TrieNode()  # 建立根節點

    # 新增單字到 Trie 中
    def addWord(self, word: str) -> None:
        current = self.root  # 從根節點開始
        for char in word:    # 逐字元處理單詞 -> Time: O(N)
            if char not in current.children:
                current.children[char] = TrieNode()  # 若無此字元節點，則創建新節點 -> Space: O(N)
            current = current.children[char]  # 移動到下一個節點

        current.end = True  # 單詞插入完成，標記結尾

    # 搜尋單字，支援 "." 作為萬用字元
    def search(self, word: str) -> bool:
        # 使用 DFS 遞迴搜尋，index 表示目前處理到 word 的哪個位置
        # 最壞情況下每個搜尋都是 "."，會遍歷整棵樹或展開所有分支 -> Time: O(min(M * L, Σ^N))
        def dfs(index, node): # Space: O(N) 遞迴棧空間
            current = node  # 從目前節點開始搜尋
            for i in range(index, len(word)): 
                char = word[i]
                if char == ".":  # 如果是萬用字元
                    # 嘗試所有子節點，看是否能繼續匹配成功
                    for sub_node in current.children.values():
                        if dfs(index=i+1, node=sub_node):  # 從下一個字元開始搜尋
                            return True  # 只要有一條路成功就回傳 True
                    return False  # 所有子節點都不成功就回傳 False
                else:
                    if char not in current.children:
                        return False  # 字母不存在於當前節點，直接回傳 False
                    current = current.children[char]  # 移動到下一個節點
                    
            return current.end  # 所有字母都處理完後，回傳是否為單字結尾
        
        # 從根節點開始遞迴搜尋
        return dfs(index=0, node=self.root)

In [2]:
# Input
# ["WordDictionary","addWord","addWord","addWord","search","search","search","search"]
# [[],["bad"],["dad"],["mad"],["pad"],["bad"],[".ad"],["b.."]]
# Output
# [null,null,null,null,false,true,true,true]

wordDictionary = WordDictionary()
print(wordDictionary.addWord("bad"))
print(wordDictionary.addWord("dad"))
print(wordDictionary.addWord("mad"))
print(wordDictionary.search("pad")) # return False
print(wordDictionary.search("bad")) # return True
print(wordDictionary.search(".ad")) # return True
print(wordDictionary.search("b..")) # return True

None
None
None
False
True
True
True
